In [ ]:
import numpy as np
import pandas as pd
import pickle
import sys
import os
import torch

sys.path.insert(0,'..')

from src.text_features import decode_labels, CombinedVectorizer
from src.ffnn import FFNN

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

In [ ]:
df_subm = pd.read_csv('subm2.csv', sep=';')

texts = df_subm['Text'].fillna('').tolist()

print(f'amostras a classificar: {len(texts)}')
print(df_subm.head(3))

In [ ]:
with open('../vectorizers/vectorizer_dnn_v2.pkl', 'rb') as f:
    vec = pickle.load(f)

X_subm = vec.transform(texts)
X_tensor = torch.tensor(X_subm, dtype=torch.float32).to(device)

print(f'Dados transformados! (shape: {X_tensor.shape})')

In [ ]:
input_dim = X_tensor.shape[1]
model = FFNN(input_dim=input_dim, n_classes=5,
             topology=[512, 256, 128], dropout=0.4).to(device)
model.load_state_dict(torch.load('../models/model_dnn_v2.pt', map_location=device))
model.eval()

with torch.no_grad():
    outputs   = model(X_tensor)
    preds_idx = torch.argmax(outputs, dim=1).cpu().numpy()

print('Inferência concluída!')

In [ ]:
preds_raw = decode_labels(preds_idx)

label_mapping = {
    'human':     'Human',
    'anthropic': 'Anthropic',
    'google':    'Google',
    'openai':    'OpenAI',
    'meta':      'Meta'
}
preds_formatted = [label_mapping[l] for l in preds_raw]

df_subm['Labels'] = preds_formatted
print(df_subm['Labels'].value_counts())

filename = 'subm2-g2-MEI-A.csv'
df_subm.to_csv(filename, index=False)
print(f"\nFicheiro '{filename}' guardado com sucesso!")
print(df_subm.head())